# Risk-agirlikli A* - Kapanma-farkinda rota

**Referans satiri.** Sert kapatma (25 m) + hasar yakinligina gore kenar cezasi.  
Butun kiyaslar `common.py` uzerinden AYNI graf / OD / kapanma.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/06_rota
    %pip install -q osmnx pyrosm networkx
print('cwd:', os.getcwd())

## 1. Graf + kapanma + OD ciftleri

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import common as C
C.CONFIG.update(dict(
    place='Antakya, Hatay, Turkey',   # veya 'Kahramanmaras, Turkey'
    n_od_pairs=120,                    # senaryo: Antakya 126, Maras 35
))
G = C.load_graph()
closures = C.load_closures()
od = C.sample_od_pairs(G, C.CONFIG['n_od_pairs'], C.CONFIG['seed'])
print('dugum/kenar:', G.number_of_nodes(), G.number_of_edges(), ' kapanma:', len(closures))

## 2. Tam model (hard + soft) ile rota

In [ ]:
H = C.apply_damage(G, closures, C.CONFIG, mode='full')
metrics, routes = C.evaluate_routes(H, od, algo='astar')
metrics

## 3. Referans yollari sakla (rota degisim orani icin)

In [ ]:
baseline_paths = [r['path'] if r['ok'] else None for r in routes]
import pickle; pickle.dump(baseline_paths, open('data/baseline_paths.pkl', 'wb'))
C.save_result('risk_weighted_astar', metrics, extra={'mode': 'full'})

## 4. 80/80 erisilebilirlik egrisi

In [ ]:
acc = C.accessibility_80_80(G, closures, od, C.CONFIG)
print('curve:', acc['curve'])
print('%80 erisim korunan max bozulma:', acc['max_disruption_at_80pct_access'])